# 02 · Capa Silver: limpieza y preparación

**Qué hace:** toma la tabla Bronze (todo texto) y produce una tabla limpia: columnas con nombres claros, tipos correctos,
valores imposibles convertidos en nulos, categorías en mayúsculas y los dos indicadores por fila.

Usa operaciones que pide el docente: `select`, `filter`, `withColumnRenamed`, `na.fill`, `describe` y `groupBy`.
Todo está dentro de `limpiar_a_silver` (notebook 00), para que la misma limpieza se aplique a datos reales y sintéticos.

In [ ]:
%run ./00_utilidades

In [ ]:
df_bronze = leer_tabla(T_BRONZE)
df_silver = limpiar_a_silver(df_bronze)
df_silver.printSchema()
display(df_silver.limit(10))

## Guardar Silver
Particionamos por `anio`: Spark crea una carpeta por año y, al filtrar por año, lee solo esa carpeta.

In [ ]:
guardar_tabla(df_silver, T_SILVER_1X, particion="anio")
silver = leer_tabla(T_SILVER_1X)
n_bronze, n_silver = df_bronze.count(), silver.count()
print(f"Bronze: {n_bronze:,} | Silver: {n_silver:,} | Retención: {n_silver / n_bronze * 100:.2f} %")

## Reporte de calidad
¿Cuántos valores tuvo que corregir la limpieza? (nulos = valores que eran imposibles o inválidos)

In [ ]:
columnas = ["peso_gramos", "talla_cm", "semanas_gestacion", "edad_madre"]
nulos = silver.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in columnas]).collect()[0].asDict()
ign = silver.select([F.sum(F.when(F.col(c) == "IGNORADO", 1).otherwise(0)).alias(c)
                     for c in ["financiador_parto", "atiende_parto", "num_embarazos", "estado_civil"]]).collect()[0].asDict()
display(spark.createDataFrame([(k, int(v), "nulo") for k, v in nulos.items()] +
                              [(k, int(v), "IGNORADO") for k, v in ign.items()], ["columna", "filas_corregidas", "tipo"]))

In [ ]:
display(silver.select("peso_gramos", "talla_cm", "semanas_gestacion", "edad_madre").describe())

In [ ]:
display(silver.groupBy("anio").count().orderBy("anio"))

## Validación: los indicadores deben coincidir con el Data Mart de SQL Server

In [ ]:
display(calcular_kpis(silver))

**Resultado esperado:** I1 = 1,048,575 · I2 = 6.42 % · I3 = 13.16 %.
Si tu dashboard de Power BI muestra otros valores (por ejemplo 1,046 mill. y 6.45 %), el Data Mart todavía no tiene la corrección de los `LEFT JOIN`.